Convexity checker

In [6]:
import math 
import numpy as np 
import random 


In [7]:
def check_convexity(f, dim, bounds=(-5, 5), samples=1000):
    violations = 0
    for _ in range(samples):
        x = [random.uniform(*bounds) for _ in range(dim)]
        y = [random.uniform(*bounds) for _ in range(dim)]
        t = random.uniform(0, 1)
        mid = [t * xi + (1 - t) * yi for xi, yi in zip(x, y)]
        lhs = f(mid)
        rhs = t * f(x) + (1 - t) * f(y)
        if lhs > rhs + 1e-10:
            violations += 1
    return violations == 0, violations

In [1]:
# Convex: quadratic with positive definite Hessian
f_convex   = lambda v: v[0]**2 + v[1]**2                    # bowl
f_convex2  = lambda v: (v[0] - 1)**2 + 2*(v[1] + 3)**2 + 5  # shifted bowl
f_convex3  = lambda v: abs(v[0]) + abs(v[1])                # L1 (convex, kink)
f_convex4  = lambda v: max(v[0], v[1])                      # max of linear (convex)
f_convex5  = lambda v: 3*v[0]**2 + v[0]*v[1] + 2*v[1]**2    # PD quadratic
f_linear   = lambda v: 2*v[0] - 3*v[1] + 7                  # affine (both convex & concave)

In [2]:
# Non-convex
f_concave  = lambda v: -(v[0]**2 + v[1]**2)                 # inverted bowl
f_saddle   = lambda v: v[0]**2 - v[1]**2                    # saddle
f_sin      = lambda v: math.sin(v[0]) + math.sin(v[1])      # wavy
f_quartic  = lambda v: -(v[0]**4 + v[1]**4)                 # concave quartic
f_circle   = lambda v: -math.sqrt(v[0]**2 + v[1]**2 + 1)    # concave-ish

In [4]:
tests = [
    ("bowl (convex)",        f_convex,  True),
    ("shifted bowl",         f_convex2, True),
    ("L1 norm",              f_convex3, True),
    ("max(x,y)",             f_convex4, True),
    ("PD quadratic",         f_convex5, True),
    ("affine",               f_linear,  True),
    ("inverted bowl",        f_concave, False),
    ("saddle x^2 - y^2",     f_saddle,  False),
    ("sin(x)+sin(y)",        f_sin,     False),
    ("concave quartic",      f_quartic, False),
    ("-sqrt(x^2+y^2+1)",     f_circle,  False),
]

In [8]:
random.seed(0)
for name, f, expected in tests:
    ok, v = check_convexity(f, dim=2, samples=2000)
    status = "PASS" if ok == expected else "FAIL"
    print(f"[{status}] {name:22s} convex={ok!s:5s} violations={v}")

[PASS] bowl (convex)          convex=True  violations=0
[PASS] shifted bowl           convex=True  violations=0
[PASS] L1 norm                convex=True  violations=0
[PASS] max(x,y)               convex=True  violations=0
[PASS] PD quadratic           convex=True  violations=0
[PASS] affine                 convex=True  violations=0
[PASS] inverted bowl          convex=False violations=2000
[PASS] saddle x^2 - y^2       convex=False violations=999
[PASS] sin(x)+sin(y)          convex=False violations=1001
[PASS] concave quartic        convex=False violations=2000
[PASS] -sqrt(x^2+y^2+1)       convex=False violations=2000


Newton's method for 2D

In [9]:
def newtons_method(f, grad_f, hessian_f, x0, steps=50, tol=1e-12):
    x = list(x0)
    history = [x[:]]
    for _ in range(steps):
        g = grad_f(x)
        H = hessian_f(x)
        det = H[0][0] * H[1][1] - H[0][1] * H[1][0]
        if abs(det) < 1e-15:
            break
        H_inv = [
            [H[1][1] / det, -H[0][1] / det],
            [-H[1][0] / det, H[0][0] / det],
        ]
        dx = [
            H_inv[0][0] * g[0] + H_inv[0][1] * g[1],
            H_inv[1][0] * g[0] + H_inv[1][1] * g[1],
        ]
        x = [x[0] - dx[0], x[1] - dx[1]]
        history.append(x[:])
        if sum(gi ** 2 for gi in g) < tol:
            break
    return history

In [10]:
def f(v):       return (v[0]-1)**4 + (v[1]+2)**4 + v[0]**2 + v[1]**2
def grad(v):    return [4*(v[0]-1)**3 + 2*v[0], 4*(v[1]+2)**3 + 2*v[1]]
def hess(v):    return [[12*(v[0]-1)**2 + 2, 0],
                        [0, 12*(v[1]+2)**2 + 2]]

hist = newtons_method(f, grad, hess, [3.0, 3.0], steps=20)
for i, p in enumerate(hist):
    print(i, [f"{v:.6f}" for v in p], "f =", f(p))

0 ['3.000000', '3.000000'] f = 659.0
1 ['2.240000', '1.324503'] f = 131.28995026559252
2 ['1.648030', '0.213122'] f = 26.927275858087896
3 ['1.025157', '-0.507322'] f = 6.272692816256402
4 ['0.003846', '-0.934945'] f = 3.145572200809465
5 ['0.287594', '-1.124712'] f = 2.192219970128907
6 ['0.395262', '-1.163387'] f = 2.133333612423508
7 ['0.409993', '-1.164876'] f = 2.1326212444936252
8 ['0.410245', '-1.164878'] f = 2.132621046960211
9 ['0.410245', '-1.164878'] f = 2.132621046960195
